# SignBridge — Master Colab Training Pipeline

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/joshiaradhya/SignBridge/blob/main/notebooks/00_master_training_pipeline.ipynb)

Run **Runtime → Run all** once. The pipeline mounts Google Drive, prepares signer-safe manifests, builds the YOLO data, extracts landmarks, resumes temporal-model training from saved checkpoints, exports ONNX models, and writes a durable status report.

All durable files live in `MyDrive/SignBridge_Data/pipeline/`; `/content/SignBridge` is only a disposable code checkout. On a first run, the notebook can download the official INCLUDE source videos into `MyDrive/SignBridge_Data/include50/`. ASL Citizen is not downloaded automatically because its official archive is 42.8 GB and has a research-only license.

In [ ]:
# One-time runtime setup. The data and all outputs remain in Drive.
!git clone https://github.com/joshiaradhya/SignBridge.git /content/SignBridge || (cd /content/SignBridge && git pull --ff-only)
%cd /content/SignBridge
!pip install -q -r ml/colab-requirements.txt
!pip install -q ultralytics

In [ ]:
# Configuration — change these only when you intentionally want to rebuild a stage.
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
DATA_DIR = Path('/content/drive/MyDrive/SignBridge_Data')
PIPELINE_DIR = DATA_DIR / 'pipeline'
MANIFEST_DIR = PIPELINE_DIR / 'manifests'
FEATURE_DIR = PIPELINE_DIR / 'features'
CHECKPOINT_DIR = PIPELINE_DIR / 'checkpoints'
RUN_DIR = PIPELINE_DIR / 'runs'
ARTIFACT_DIR = PIPELINE_DIR / 'artifacts'
STATE_PATH = PIPELINE_DIR / 'pipeline_state.json'

# Set one or more flags to True only to deliberately redo existing work.
FORCE_REBUILD_YOLO = False
FORCE_REEXTRACT = False
FORCE_RETRAIN = False
FORCE_REEXPORT = False

# First-run data import. This downloads the official CC-BY INCLUDE archive to Drive.
# Set to False only if you will provide your own INCLUDE videos.
DOWNLOAD_INCLUDE = True
MAX_YOLO_VIDEOS = 100
YOLO_EPOCHS = 30
TEMPORAL_EPOCHS = 50

for directory in (DATA_DIR, PIPELINE_DIR, MANIFEST_DIR, FEATURE_DIR, CHECKPOINT_DIR, RUN_DIR, ARTIFACT_DIR):
    directory.mkdir(parents=True, exist_ok=True)
print(f'Persistent pipeline location: {PIPELINE_DIR}')

In [ ]:
# Download the official INCLUDE videos once, directly into the Drive-backed source folder.
# The marker makes this safe to resume after a Colab disconnect.
if DOWNLOAD_INCLUDE:
    import json
    import tempfile
    import urllib.request
    import zipfile

    include_dir = DATA_DIR / 'include50'
    marker_path = include_dir / '.include_download_state.json'
    existing_videos = list(include_dir.rglob('*.mp4')) + list(include_dir.rglob('*.mov'))
    if existing_videos:
        print(f'INCLUDE source already present ({len(existing_videos)} videos); skipping download.')
    else:
        include_dir.mkdir(parents=True, exist_ok=True)
        state = json.loads(marker_path.read_text()) if marker_path.exists() else {'completed_archives': []}
        with urllib.request.urlopen('https://zenodo.org/api/records/4010759') as response:
            files = json.load(response)['files']
        with tempfile.TemporaryDirectory(prefix='include-', dir='/content') as temp_dir:
            for item in files:
                name = item['key']
                if name in state['completed_archives']:
                    continue
                archive_path = Path(temp_dir) / name
                print(f'Downloading {name}…')
                urllib.request.urlretrieve(item['links']['self'], archive_path)
                if archive_path.suffix.lower() == '.zip':
                    with zipfile.ZipFile(archive_path) as archive:
                        destination = include_dir.resolve()
                        for member in archive.infolist():
                            target = (destination / member.filename).resolve()
                            if target != destination and destination not in target.parents:
                                raise RuntimeError(f'Unsafe archive member: {member.filename}')
                        archive.extractall(destination)
                state['completed_archives'].append(name)
                marker_path.write_text(json.dumps(state, indent=2) + '\n')
        video_count = sum(1 for path in include_dir.rglob('*') if path.suffix.lower() in {'.mp4', '.mov', '.avi', '.mkv', '.webm'})
        if not video_count:
            raise RuntimeError('INCLUDE download finished but no videos were extracted.')
        print(f'INCLUDE import complete: {video_count} videos in {include_dir}')

In [ ]:
# Master pipeline. Failures are recorded per stage, so a missing optional dataset does not erase completed work.
import json
import shutil
import traceback
from datetime import datetime, timezone

from ml.signbridge.datasets import (
    DatasetManifest, export_yolo_yaml, prepare_asl_citizen_manifest,
    prepare_include_manifest, sample_yolo_frames, verify_signer_isolation,
)
from ml.signbridge.extraction import process_manifest_to_npz
from ml.signbridge.training import SignModelTrainer, TrainingConfig, evaluate_test_set
from ml.signbridge.export import export_to_onnx

state = json.loads(STATE_PATH.read_text()) if STATE_PATH.exists() else {'stages': {}}
state['started_at'] = datetime.now(timezone.utc).isoformat()

def save_state():
    state['updated_at'] = datetime.now(timezone.utc).isoformat()
    STATE_PATH.write_text(json.dumps(state, indent=2) + '\n')

def run_stage(name, action):
    print(f'\n{'=' * 16} {name} {'=' * 16}')
    state['stages'][name] = {'status': 'running', 'started_at': datetime.now(timezone.utc).isoformat()}
    save_state()
    try:
        result = action()
        state['stages'][name] = {
            'status': 'completed', 'finished_at': datetime.now(timezone.utc).isoformat(), 'result': result
        }
        print(f'✓ {name} complete')
        return result
    except Exception as exc:
        state['stages'][name] = {
            'status': 'failed', 'finished_at': datetime.now(timezone.utc).isoformat(),
            'error': str(exc), 'traceback': traceback.format_exc(limit=8)
        }
        print(f'✗ {name} failed: {exc}')
        return None
    finally:
        save_state()

def video_files(root):
    suffixes = {'.mp4', '.mov', '.avi', '.mkv', '.webm'}
    return sorted(path for path in root.rglob('*') if path.suffix.lower() in suffixes)

def prepare_manifests():
    results = {}
    include_dir = DATA_DIR / 'include50'
    if video_files(include_dir):
        manifest = prepare_include_manifest(include_dir, val_ratio=0.15, test_ratio=0.15)
        valid, message = verify_signer_isolation(manifest)
        if not valid: raise ValueError(message)
        destination = MANIFEST_DIR / 'include50_manifest.json'
        manifest.save(destination)
        results['isl_manifest'] = str(destination)
    else:
        results['isl_manifest'] = 'skipped: no INCLUDE-50 videos'

    asl_dir = DATA_DIR / 'asl_citizen'
    metadata, videos = asl_dir / 'asl_citizen_metadata.csv', asl_dir / 'videos'
    if metadata.is_file() and video_files(videos):
        manifest = prepare_asl_citizen_manifest(metadata, videos, val_ratio=0.15, test_ratio=0.15)
        valid, message = verify_signer_isolation(manifest)
        if not valid: raise ValueError(message)
        destination = MANIFEST_DIR / 'asl_citizen_manifest.json'
        manifest.save(destination)
        results['asl_manifest'] = str(destination)
    else:
        results['asl_manifest'] = 'skipped: ASL metadata or videos missing'
    if not any(str(value).endswith('.json') for value in results.values()):
        raise FileNotFoundError(f'No source datasets found under {DATA_DIR}')
    return results

def build_yolo_dataset():
    yolo_root, yolo_yaml = PIPELINE_DIR / 'yolo_dataset', PIPELINE_DIR / 'yolo_dataset' / 'data.yaml'
    existing = yolo_yaml.exists() and all((yolo_root / group / split).exists() for group in ('images', 'labels') for split in ('train', 'val'))
    if existing and not FORCE_REBUILD_YOLO:
        return {'status': 'reused', 'yaml': str(yolo_yaml)}
    source_videos = [path for path in video_files(DATA_DIR) if yolo_root not in path.parents][:MAX_YOLO_VIDEOS]
    if len(source_videos) < 2:
        raise FileNotFoundError(f'Need at least two videos under {DATA_DIR} to create train and validation splits')
    if FORCE_REBUILD_YOLO and yolo_root.exists():
        shutil.rmtree(yolo_root)
    stats = sample_yolo_frames(source_videos, yolo_root, frames_per_video=3)
    export_yolo_yaml(yolo_yaml, yolo_root)
    return {'status': 'created', 'yaml': str(yolo_yaml), 'frames': stats}

def validate_yolo_dataset():
    yolo_root, yolo_yaml = PIPELINE_DIR / 'yolo_dataset', PIPELINE_DIR / 'yolo_dataset' / 'data.yaml'
    if not yolo_yaml.exists(): raise FileNotFoundError('YOLO data.yaml was not created')
    counts = {}
    for split in ('train', 'val'):
        images = list((yolo_root / 'images' / split).glob('*'))
        labels = list((yolo_root / 'labels' / split).glob('*.txt'))
        annotated = sum(bool(path.read_text().strip()) for path in labels)
        counts[split] = {'images': len(images), 'labels': len(labels), 'annotated': annotated}
        if not images or len(labels) != len(images) or annotated != len(labels):
            raise ValueError(f'Invalid {split} YOLO split: {counts[split]}')
    return counts

def train_yolo():
    from ultralytics import YOLO
    yolo_yaml = PIPELINE_DIR / 'yolo_dataset' / 'data.yaml'
    output_dir = RUN_DIR / 'yolo' / 'person_hand_face'
    latest = output_dir / 'weights' / 'last.pt'
    if latest.exists() and not FORCE_RETRAIN:
        model = YOLO(str(latest))
        model.train(resume=True)
    else:
        model = YOLO('yolov8n.pt')
        model.train(data=str(yolo_yaml), epochs=YOLO_EPOCHS, imgsz=640, batch=16, device=0, project=str(RUN_DIR / 'yolo'), name='person_hand_face')
    best = output_dir / 'weights' / 'best.pt'
    if best.exists(): shutil.copy2(best, ARTIFACT_DIR / 'sign_yolo_v1.pt')
    return {'best_checkpoint': str(best)}

def extract_features():
    results = {}
    for language, manifest_name in (('isl', 'include50_manifest.json'), ('asl', 'asl_citizen_manifest.json')):
        manifest_path, feature_path = MANIFEST_DIR / manifest_name, FEATURE_DIR / f'{language}_features_v2.npz'
        if not manifest_path.exists():
            results[language] = 'skipped: source manifest unavailable'
        elif feature_path.exists() and not FORCE_REEXTRACT:
            results[language] = 'reused: ' + str(feature_path)
        else:
            results[language] = process_manifest_to_npz(DatasetManifest.load(manifest_path), feature_path)
    if not any((FEATURE_DIR / f'{language}_features_v2.npz').exists() for language in ('isl', 'asl')):
        raise RuntimeError('No feature archives were produced')
    return results

def train_temporal_models():
    import torch
    results = {}
    for language in ('isl', 'asl'):
        features = FEATURE_DIR / f'{language}_features_v2.npz'
        if not features.exists():
            results[language] = 'skipped: feature archive unavailable'
            continue
        checkpoint_dir, output_dir = CHECKPOINT_DIR / language, RUN_DIR / language
        best = checkpoint_dir / 'best_model.pt'
        if best.exists() and not FORCE_RETRAIN:
            results[language] = 'reused: ' + str(best)
            continue
        config = TrainingConfig(epochs=TEMPORAL_EPOCHS, batch_size=32, device='cuda' if torch.cuda.is_available() else 'cpu')
        trainer = SignModelTrainer(config, output_dir=output_dir, checkpoint_dir=checkpoint_dir)
        trainer.fit(features, resume_latest=not FORCE_RETRAIN)
        results[language] = evaluate_test_set(best, features, output_dir / 'eval')
    return results

def export_models():
    results = {}
    for language in ('isl', 'asl'):
        checkpoint = CHECKPOINT_DIR / language / 'best_model.pt'
        model_path, metadata_path = ARTIFACT_DIR / f'{language}-bilstm-v1.onnx', ARTIFACT_DIR / f'{language}-metadata.json'
        if not checkpoint.exists():
            results[language] = 'skipped: best checkpoint unavailable'
        elif model_path.exists() and metadata_path.exists() and not FORCE_REEXPORT:
            results[language] = 'reused: ' + str(model_path)
        else:
            results[language] = export_to_onnx(checkpoint, model_path, metadata_path)
    archive = shutil.make_archive(str(PIPELINE_DIR / 'signbridge_onnx_models'), 'zip', ARTIFACT_DIR)
    results['archive'] = archive
    return results

run_stage('manifests', prepare_manifests)
run_stage('yolo_dataset', build_yolo_dataset)
run_stage('yolo_validation', validate_yolo_dataset)
run_stage('yolo_training', train_yolo)
run_stage('feature_extraction', extract_features)
run_stage('temporal_training', train_temporal_models)
run_stage('onnx_export', export_models)

failures = [name for name, result in state['stages'].items() if result['status'] == 'failed']
print('\nPipeline report:', STATE_PATH)
print('Failed stages:' if failures else 'All runnable stages completed.', ', '.join(failures) if failures else '')
save_state()